# PPO Pairs Trading Agents

In [ ]:
import sys, os
sys.path.insert(0, '.')   # make env importable

import numpy as np
from stable_baselines3 import PPO
from env import PairsTradingEnv

import warnings; warnings.filterwarnings('ignore')


PAIR_NAME        = 'V_MA'
DATA_PATH        = f'./regime_output_{PAIR_NAME}/generated.npz'
TRANSACTION_COST = 0.001
ZSCORE_WINDOW    = 20
TOTAL_TIMESTEPS  = 250_000 
SEED             = 42   # data split + RNG
SEEDS            = [7]  # training seeds
N_ENVS           = 4
CALIB_ALPHA      = 0.05  # online calibration: EMA rate for the per-regime mean

np.random.seed(SEED)

## 1. Load Generated Data

In [ ]:
d = np.load(DATA_PATH)
spreads = d['spreads'].astype(np.float32)
regimes = d['regimes'].astype('int8')

N, T = spreads.shape
print(f'Loaded: {N} episodes x {T} steps')
print(f'Spread mean={spreads.mean():.4f}  std={spreads.std():.4f}')

# 80/20 split
rng     = np.random.default_rng(SEED)
idx     = rng.permutation(N)
n_train = int(0.8 * N)
tr, ev  = idx[:n_train], idx[n_train:]

sp_tr, rg_tr = spreads[tr], regimes[tr]
sp_ev, rg_ev = spreads[ev], regimes[ev]
BETA = float(np.load(f'./regime_output_{PAIR_NAME}/beta.npy')[0])
print(f'Beta: {BETA:.4f}')
print(f'Train: {len(tr)}  Eval: {len(ev)}')

## 2. Train Baseline Agent (No Regimes)

In [ ]:
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import (
    EvalCallback, StopTrainingOnNoModelImprovement
)
from stable_baselines3.common.monitor import Monitor
import torch

HYPERPARAMS = dict(
    learning_rate = 3e-4,
    n_steps       = T,
    batch_size    = 64,
    n_epochs      = 10,
    gamma         = 0.99,
    gae_lambda    = 0.95,
    clip_range    = 0.2,
    ent_coef      = 0.005,
    vf_coef       = 0.5,
    max_grad_norm = 0.5,
    policy_kwargs = dict(
        net_arch      = [128, 128],
        activation_fn = torch.nn.Tanh,
        log_std_init  = -1.0,
    ),
)


import pickle as _pickle
_ou = _pickle.load(open(f'./regime_output_{PAIR_NAME}/ou_params.pkl', 'rb'))
REG_EXTRA = dict(mu_r=_ou['mu_r'], sigma_r=_ou['sigma_r'], kappa=_ou['kappa'])

def make_env(sp, rg, use_reg):
    def _fn():
        extra = REG_EXTRA if use_reg else {}
        return Monitor(PairsTradingEnv(
            sp, rg,
            use_regimes      = use_reg,
            transaction_cost = TRANSACTION_COST,
            zscore_window    = ZSCORE_WINDOW,
            calib_alpha      = CALIB_ALPHA,
            **extra,
        ))
    return _fn

eval_env_base = Monitor(PairsTradingEnv(
    sp_ev, rg_ev, use_regimes=False,
    transaction_cost=TRANSACTION_COST, zscore_window=ZSCORE_WINDOW
))


for s in SEEDS:
    save_dir = f'models_{PAIR_NAME}_sim/baseline/seed{s}'
    log_dir  = f'logs_{PAIR_NAME}_sim/baseline/seed{s}'
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir,  exist_ok=True)

    train_env_base = make_vec_env(
        make_env(sp_tr, rg_tr, False), n_envs=N_ENVS, seed=s
    )

    stop_cb_base = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals = 10,   # stop after 10 evals with no improvement
        min_evals                = 20,   # always run at least 20 evals first
        verbose                  = 1,
    )
    eval_cb_base = EvalCallback(
        eval_env_base,
        best_model_save_path = f'{save_dir}/',
        log_path             = f'{log_dir}/',
        eval_freq            = TOTAL_TIMESTEPS // 50,
        n_eval_episodes      = 50,
        deterministic        = True,
        callback_after_eval  = stop_cb_base,
        verbose              = 0,
    )

    model_base = PPO('MlpPolicy', train_env_base,
                     **HYPERPARAMS, verbose=0, seed=s)
    print(f'Training baseline agent (seed={s})...')
    model_base.learn(TOTAL_TIMESTEPS, callback=eval_cb_base, progress_bar=True)
    model_base.save(f'{save_dir}/final')

print(f'Baseline training complete for seeds {SEEDS}.')

## 3. Train Regime-Aware Agent

In [ ]:
eval_env_reg = Monitor(PairsTradingEnv(
    sp_ev, rg_ev, use_regimes=True,
    transaction_cost=TRANSACTION_COST, zscore_window=ZSCORE_WINDOW,
    calib_alpha=CALIB_ALPHA,
    **REG_EXTRA
))

HYPERPARAMS_REGIME = dict(
    learning_rate = 3e-4,
    n_steps       = T,
    batch_size    = 64,
    n_epochs      = 10,
    gamma         = 0.99,
    gae_lambda    = 0.95,
    clip_range    = 0.2,
    ent_coef      = 0.005,
    vf_coef       = 0.5,
    max_grad_norm = 0.5,
    policy_kwargs = dict(
        net_arch      = [128, 128],
        activation_fn = torch.nn.Tanh,
        log_std_init  = -1.0,
    ),
)
TOTAL_TIMESTEPS_REGIME = 250_000

for s in SEEDS:
    save_dir = f'models_{PAIR_NAME}_sim/regime/seed{s}'
    log_dir  = f'logs_{PAIR_NAME}_sim/regime/seed{s}'
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir,  exist_ok=True)

    train_env_reg = make_vec_env(
        make_env(sp_tr, rg_tr, True), n_envs=N_ENVS, seed=s
    )

    stop_cb_reg = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals = 10,
        min_evals                = 20,
        verbose                  = 1,
    )
    eval_cb_reg = EvalCallback(
        eval_env_reg,
        best_model_save_path = f'{save_dir}/',
        log_path             = f'{log_dir}/',
        eval_freq            = TOTAL_TIMESTEPS_REGIME // 50,
        n_eval_episodes      = 50,
        deterministic        = True,
        callback_after_eval  = stop_cb_reg,
        verbose              = 0,
    )

    model_reg = PPO('MlpPolicy', train_env_reg,
                    **HYPERPARAMS_REGIME, verbose=0, seed=s)
    print(f'Training regime agent (seed={s}, zadj + online-calib obs)...')
    model_reg.learn(TOTAL_TIMESTEPS_REGIME, callback=eval_cb_reg, progress_bar=True)
    model_reg.save(f'{save_dir}/final')

print(f'Regime training complete for seeds {SEEDS}.')

## 4. Learning Curves

In [ ]:
import matplotlib.pyplot as plt

def load_curves(kind):
    """Per-seed eval reward vs timesteps; aligned on the shared timestep grid.
    Returns (timesteps, mean_over_seeds, std_over_seeds, n_seeds_per_point)."""
    grid = None
    cols = []   # one column per seed, padded with NaN where the seed stopped early
    for s in SEEDS:
        f = f'logs_{PAIR_NAME}_sim/{kind}/seed{s}/evaluations.npz'
        if not os.path.exists(f):
            continue
        e = np.load(f)
        ts  = e['timesteps']
        rew = e['results'].mean(axis=1)   # mean over eval episodes -> one value per eval
        if grid is None or len(ts) > len(grid):
            grid = ts                     # longest run defines the timestep axis
        cols.append((ts, rew))

    col_arr = np.full((len(grid), len(cols)), np.nan)
    for j, (ts, rew) in enumerate(cols):
        col_arr[:len(rew), j] = rew       # grids share a prefix; pad the tail with NaN
    mean = np.nanmean(col_arr, axis=1)
    std  = np.nanstd(col_arr, axis=1)
    n    = np.sum(~np.isnan(col_arr), axis=1)
    return grid, mean, std, n

fig, ax = plt.subplots(figsize=(9, 5))
for kind, color, label in [('baseline', 'tab:gray',  'Baseline (no regimes)'),
                           ('regime',   'tab:blue',  'Regime-aware')]:
    ts, mean, std, n = load_curves(kind)
    ax.plot(ts, mean, color=color, lw=2, label=label)
    ax.fill_between(ts, mean - std, mean + std, color=color, alpha=0.18)

ax.set_xlabel('Training timesteps')
ax.set_ylabel('Mean eval episode reward')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()